# graphrag_stage1 — Jupyter Notebook Usage Guide

This notebook demonstrates how to use the **graphrag_stage1** library in a Jupyter notebook environment. The library transforms unstructured scientific text into a typed, provenance-anchored Knowledge Artifact Graph (KAG) and validated semantic frames, with optional ontology mapping to RDF/JSON-LD.

## Pipeline Overview

| Stage | Description | Output |
|-------|-------------|--------|
| **Stage 1** | Atomic fact extraction + facet typing + provenance anchoring | Knowledge Artifact Graph (KAG) |
| **Stage 2** | Semantic frame construction + grounding validation | Validated Semantic Frames |
| **Stage 3** | Ontology mapping + instance generation + RDF emission | Ontology-aligned RDF/JSON-LD |

---

## Prerequisites

- Python 3.10+
- An LLM provider (Anthropic, OpenAI, or local Ollama)
- Optional: Domain ontology (OWL/TTL) for Stage 3

## 1. Installation

Install the library with desired extras:

In [ ]:
# Core library (Stage 1 + 2) - only requires 'requests'
%pip install graphrag-stage1

# Optional: Stage 3 ontology mapping (requires rdflib, pyshacl)
# %pip install "graphrag-stage1[ontology]"

# Optional: Built-in LLM clients
# %pip install "graphrag-stage1[anthropic]"   # Anthropic Claude
# %pip install "graphrag-stage1[openai]"      # OpenAI GPT
# %pip install "graphrag-stage1[dev]"         # pytest, jsonschema for testing

# If running locally, restart kernel after installation
import sys
print(f"Python: {sys.version}")

## 2. Imports & LLM Client Setup

Choose your LLM provider. The library uses an `LLMClient` protocol — use built-in clients or implement your own.

In [ ]:
import os
import json
from pprint import pprint

# Core pipeline functions
from graphrag_stage1 import (
    run_pipeline,           # Single paragraph: Stage 1 -> 2 -> (3)
    run_paper,              # Multiple paragraphs in parallel
    analyze_paper,          # Full paper text -> split -> process
    process_paragraph,      # Stage 1 only
    stage2_pipeline,        # Stage 2 only
    validate,               # Schema validation
)

# Built-in LLM clients (require corresponding extras installed)
from graphrag_stage1 import AnthropicClient, OpenAIClient
from graphrag_stage1.llm import OllamaClient, LLMClient

# Stage 3: Ontology management
from graphrag_stage1.ontology_manager import OntologyManager
from graphrag_stage1.stage3_jsonld import build_jsonld

# ============================================================
# CHOOSE YOUR LLM CLIENT (uncomment one)
# ============================================================

# Option A: Anthropic (requires ANTHROPIC_API_KEY env var)
# client = AnthropicClient()

# Option B: OpenAI (requires OPENAI_API_KEY env var)
# client = OpenAIClient()

# Option C: Local Ollama (requires Ollama running locally)
# Set env vars: STAGE1_MODEL, STAGE2_MODEL, STAGE2_STRONGER_MODEL, OLLAMA_URL
client = OllamaClient()  # Uses env vars; good for local dev

# Option D: Custom client implementing LLMClient protocol
# class MyClient:
#     model_id = "my-model"
#     def complete(self, prompt: str, schema: dict, *, stronger: bool = False) -> dict:
#         # Your implementation here
#         pass
# client = MyClient()

print(f"Using client: {client.model_id}")

## 1. Install the Library

Install the core library (Stage 1 + 2) and optional extras for Stage 3 and LLM providers.

In [ ]:
# Core library (Stage 1 + 2) - only requires 'requests'
%pip install graphrag-stage1

# Optional: Stage 3 ontology mapping (requires rdflib, pyshacl)
# %pip install "graphrag-stage1[ontology]"

# Optional: Built-in LLM clients
# %pip install "graphrag-stage1[anthropic]"   # Anthropic Claude
# %pip install "graphrag-stage1[openai]"      # OpenAI GPT
# %pip install "graphrag-stage1[dev]"         # pytest, jsonschema for testing

## 2. Imports & LLM Client Setup

Choose your LLM provider. The library uses an `LLMClient` protocol — you can use built-in clients or implement your own.

In [ ]:
import os
import json
from pprint import pprint

# Core pipeline functions
from graphrag_stage1 import (
    run_pipeline,           # Single paragraph: Stage 1 -> 2 -> (3)
    run_paper,              # Multiple paragraphs in parallel
    analyze_paper,          # Full paper text -> split -> process
    process_paragraph,      # Stage 1 only
    stage2_pipeline,        # Stage 2 only
    validate,               # Schema validation
)

# Built-in LLM clients (require corresponding extras installed)
from graphrag_stage1 import AnthropicClient, OpenAIClient
from graphrag_stage1.llm import OllamaClient, LLMClient

# Stage 3: Ontology management
from graphrag_stage1.ontology_manager import OntologyManager
from graphrag_stage1.stage3_jsonld import build_jsonld

# ============================================================
# CHOOSE YOUR LLM CLIENT (uncomment one)
# ============================================================

# Option A: Anthropic (requires ANTHROPIC_API_KEY env var)
# client = AnthropicClient()

# Option B: OpenAI (requires OPENAI_API_KEY env var)
# client = OpenAIClient()

# Option C: Local Ollama (requires Ollama running locally)
# Set env vars: STAGE1_MODEL, STAGE2_MODEL, STAGE2_STRONGER_MODEL, OLLAMA_URL
client = OllamaClient()  # Uses env vars; good for local dev

# Option D: Custom client implementing LLMClient protocol
# class MyClient:
#     model_id = "my-model"
#     def complete(self, prompt: str, schema: dict, *, stronger: bool = False) -> dict:
#         # Your implementation here
#         pass
# client = MyClient()

print(f"Using client: {client.model_id}")

## 3. Stage 1 Only — Knowledge Artifact Graph (KAG)

Process a single paragraph to extract atomic propositions with facet typing and provenance anchoring.

In [ ]:
# Sample scientific paragraph
paragraph = """
The fused sensor data reduced positioning error by 42% because it provides a more 
complete state estimate. The Kalman filter integrates IMU measurements at 200 Hz 
with GPS updates at 1 Hz, yielding a posterior covariance trace of 0.03 m². 
However, the improvement diminishes in urban canyons where GPS multipath exceeds 15 m.
"""

# Run Stage 1 only
kag = process_paragraph(
    paragraph,
    paragraph_id="demo:p1",
    source_metadata={
        "document_id": "demo-doc",
        "page": 1,
        "source_uri": "https://example.com/paper.pdf"
    },
    client=client
)

# Inspect the Knowledge Artifact Graph structure
print("=== KAG Keys ===")
print(list(kag.keys()))

print("\n=== Propositions (atomic facts) ===")
for i, prop in enumerate(kag.get("propositions", [])):
    print(f"\n  Proposition {i+1}:")
    print(f"    Text: {prop.get('text')}")
    print(f"    Facets: {prop.get('facets')}")
    print(f"    Provenance: {prop.get('provenance')}")

print("\n=== Discourse Relations ===")
for rel in kag.get("discourse_relations", []):
    print(f"  {rel.get('type')}: {rel.get('source')} -> {rel.get('target')}")

print("\n=== Coreference Clusters ===")
for cluster in kag.get("coreference_clusters", []):
    print(f"  Cluster {cluster.get('cluster_id')}: {cluster.get('mentions')}")

# Optional: Validate against JSON Schema
# validate(kag)  # Requires: pip install "graphrag-stage1[validation]"

## 4. Stage 1 → Stage 2 — Semantic Frames

Stage 2 converts propositions into structured semantic frames with typed slots, grounding checks, and confidence scores.

In [ ]:
# Run Stage 1 -> Stage 2 on the same paragraph
stage2_result = stage2_pipeline(kag, client=client, concurrency=4)

print("=== Stage 2 Output Keys ===")
print(list(stage2_result.keys()))

print("\n=== Semantic Frames ===")
for i, frame in enumerate(stage2_result.get("frames", [])):
    print(f"\n  Frame {i+1}: {frame.get('frame_type')}")
    print(f"    Confidence: {frame.get('confidence'):.3f}")
    print(f"    Grounded: {frame.get('grounded')}")
    print(f"    Slots:")
    for slot_name, slot_value in frame.get("slots", {}).items():
        print(f"      {slot_name}: {slot_value}")
    print(f"    Evidence: {frame.get('evidence', {}).get('source_statement', '')[:100]}...")

print("\n=== Frame-Level Provenance ===")
for frame in stage2_result.get("frames", []):
    prov = frame.get("provenance", {})
    print(f"  Statement {frame.get('statement_id')}: doc={prov.get('document_id')}, page={prov.get('page')}, chars={prov.get('char_start')}-{prov.get('char_end')}")

## 5. Stage 3 — Ontology Mapping & RDF/JSON-LD Output

Stage 3 maps validated semantic frames to your domain ontology (BFO/CCO/IAO + your domain) and emits RDF triples + JSON-LD. **Requires `[ontology]` extra and a domain ontology file.**

In [ ]:
# Load ontology manager (requires ontologies/ directory with core + domain ontologies)
# Expected structure:
# ontologies/
#   ├── bfo.owl, iao.owl, ro.owl, CommonCoreOntologiesMerged.ttl
#   ├── Alignment/stage2-alignment-v1.0.ttl
#   ├── Alignment/stage3-publication-shapes.ttl
#   ├── Domain/ino_merged.owl   <-- YOUR domain ontology
#   └── Relations/ro.owl

ontology_root = "ontologies"  # Adjust path as needed
domain_ontology = "Domain/ino_merged.owl"  # Your domain ontology

try:
    ontology = OntologyManager(ontology_root=ontology_root).load_all(domain_ontology=domain_ontology)
    print(f"Loaded ontology: {len(ontology.loaded_files)} files")
    print(f"Classes indexed: {len(ontology.class_index)}")
    print(f"Object properties: {len(ontology.object_property_index)}")
    print(f"Datatype properties: {len(ontology.datatype_property_index)}")
    
    # Run full pipeline with Stage 3
    result = run_pipeline(
        paragraph,
        client=client,
        paragraph_id="demo:p1",
        source_metadata={"document_id": "demo-doc", "page": 1},
        ontology=ontology  # <-- Triggers Stage 3
    )
    
    stage3 = result["stage3"]
    print("\n=== Stage 3 Output Keys ===")
    print(list(stage3.keys()))
    
    print("\n=== Mapped Frames ===")
    for frame in stage3.get("mapped_frames", []):
        print(f"  Frame: {frame.get('frame_type')} -> {frame.get('frame_instance', {}).get('rdf_type')}")
        print(f"    Mapping status: {frame.get('mapping_status')}")
        print(f"    Individuals: {len(frame.get('ontology_individuals', []))}")
        print(f"    Object property assertions: {len(frame.get('object_property_assertions', []))}")
        print(f"    Datatype property assertions: {len(frame.get('datatype_property_assertions', []))}")
    
    # Generate JSON-LD for graph database ingestion
    jsonld = build_jsonld(stage3)
    print(f"\n=== JSON-LD Generated ===")
    print(f"  @context keys: {list(jsonld.get('@context', {}).keys())}")
    print(f"  @graph nodes: {len(jsonld.get('@graph', []))}")
    
    # Save JSON-LD for Neptune/GraphDB import
    with open("output_stage3.jsonld", "w") as f:
        json.dump(jsonld, f, indent=2)
    print("\nSaved JSON-LD to output_stage3.jsonld")
    
    # Also serialize to RDF (Turtle) for SPARQL endpoints
    from graphrag_stage1.stage3_neptune_export import serialize_to_turtle
    turtle_output = serialize_to_turtle(stage3)
    with open("output_stage3.ttl", "w") as f:
        f.write(turtle_output)
    print("Saved Turtle RDF to output_stage3.ttl")

except FileNotFoundError as e:
    print(f"Ontology files not found: {e}")
    print("Expected structure: ontologies/ with core + domain ontologies")
except ImportError:
    print("Stage 3 requires: pip install 'graphrag-stage1[ontology]'")

## 6. Batch Processing — Full Paper (Multiple Paragraphs)

Process an entire paper by splitting into paragraphs and running in parallel with concurrency control.

In [ ]:
# Sample multi-paragraph paper text
paper_text = """
The fused sensor data reduced positioning error because it gives a more complete state estimate. 
Kalman filtering combines GPS and IMU measurements to produce a fused trajectory with lower variance.

Experimental results show that the fused solution achieves 0.5 meter accuracy in urban canyons. 
This represents a 40% improvement over GPS-only positioning. The improvement is attributed to 
the IMU bridging GPS outages during signal blockage.

Future work will evaluate multi-constellation GNSS fusion with tighter coupling. 
The proposed architecture supports adding new sensor modalities without redesign.
"""

# Option 1: Use analyze_paper (convenience function - splits on blank lines)
results = analyze_paper(
    paper_text,
    client=client,
    max_concurrency=8,        # Global rate limit
    stage2_concurrency=4,     # Per-paragraph fan-out
    ontology=ontology if 'ontology' in locals() else None,
)

print(f"Processed {len(results)} paragraphs")
for i, r in enumerate(results):
    if "error" in r:
        print(f"  Para {i}: ERROR - {r['error']}")
    else:
        s1_props = len(r["stage1"].get("propositions", []))
        s2_frames = len(r["stage2"].get("frames", []))
        print(f"  Para {i}: {s1_props} propositions -> {s2_frames} frames")

# Option 2: Manual paragraph control with run_paper
paragraphs = [
    {"text": p.strip(), "paragraph_id": f"paper:p{i+1}"}
    for i, p in enumerate(paper_text.strip().split("\n\n"))
    if p.strip()
]

results2 = run_paper(
    paragraphs,
    client=client,
    max_concurrency=8,
    stage2_concurrency=4,
    ontology=ontology if 'ontology' in locals() else None,
)

# Aggregate all Stage 3 output for full-paper knowledge graph
if 'ontology' in locals():
    all_mapped_frames = []
    all_individuals = {}
    for r in results2:
        if "stage3" in r:
            all_mapped_frames.extend(r["stage3"].get("mapped_frames", []))
            for ind in r["stage3"].get("ontology_individuals", []):
                all_individuals[ind["instance_id"]] = ind
    
    print(f"\n=== Full Paper Aggregation ===")
    print(f"Total mapped frames: {len(all_mapped_frames)}")
    print(f"Unique ontology individuals: {len(all_individuals)}")
    
    # Build combined JSON-LD
    combined_stage3 = {
        "mapped_frames": all_mapped_frames,
        "ontology_individuals": list(all_individuals.values()),
        "pipeline_version": results2[0]["stage3"]["pipeline_version"] if results2 else "1.0",
        "model": results2[0]["stage3"]["model"] if results2 else "unknown"
    }
    combined_jsonld = build_jsonld(combined_stage3)
    with open("full_paper_kg.jsonld", "w") as f:
        json.dump(combined_jsonld, f, indent=2)
    print("Saved full-paper KG to full_paper_kg.jsonld")

## 7. Validation & Schema Checking

Validate Stage 1 and Stage 2 outputs against JSON Schemas (requires `pip install "graphrag-stage1[validation]"`).

In [ ]:
from graphrag_stage1 import validate
from graphrag_stage1.validation import validate_stage1, validate_stage2

# Validate Stage 1 output
stage1_result = result["stage1"]
try:
    validate_stage1(stage1_result)
    print("✓ Stage 1 validation passed")
except Exception as e:
    print(f"✗ Stage 1 validation failed: {e}")

# Validate Stage 2 output
stage2_result = result["stage2"]
try:
    validate_stage2(stage2_result)
    print("✓ Stage 2 validation passed")
except Exception as e:
    print(f"✗ Stage 2 validation failed: {e}")

# Convenience function validates both
try:
    validate(stage1_result)  # Auto-detects stage
    validate(stage2_result)
    print("✓ All validations passed")
except Exception as e:
    print(f"Validation error: {e}")

# Inspect schema versions
print(f"\nStage 1 schema version: {stage1_result.get('pipeline_version')}")
print(f"Stage 2 schema version: {stage2_result.get('pipeline_version')}")
print(f"Model used: {stage1_result.get('model')}")

## 8. Exploration & Debugging

Inspect intermediate outputs, provenance, and model calls.

In [ ]:
# Inspect Stage 1 KAG structure
kag = result["stage1"]
print("=== Stage 1: Knowledge Artifact Graph ===")
print(f"Paragraph ID: {kag.get('paragraph_id')}")
print(f"Propositions: {len(kag.get('propositions', []))}")
print(f"Discourse relations: {len(kag.get('discourse_relations', []))}")
print(f"Mention clusters: {len(kag.get('mention_clusters', []))}")

# Show first proposition with full provenance
if kag.get("propositions"):
    prop = kag["propositions"][0]
    print(f"\n--- Proposition 0 ---")
    print(f"  ID: {prop.get('id')}")
    print(f"  Text: {prop.get('text')}")
    print(f"  Facets: {prop.get('facets')}")
    print(f"  Provenance: {prop.get('provenance')}")
    print(f"  Confidence: {prop.get('confidence')}")

# Inspect Stage 2 frames
frames = result["stage2"].get("frames", [])
print(f"\n=== Stage 2: Semantic Frames ({len(frames)}) ===")
for i, frame in enumerate(frames[:3]):
    print(f"\n--- Frame {i} ---")
    print(f"  Type: {frame.get('frame_type')}")
    print(f"  Slots: {list(frame.get('slots', {}).keys())}")
    print(f"  Grounding: {frame.get('grounding', {}).get('status')}")
    print(f"  Confidence: {frame.get('confidence')}")
    print(f"  Evidence: {frame.get('evidence', {}).get('source_statement', '')[:80]}...")

# Inspect Stage 3 mapping details
if "stage3" in result:
    stage3 = result["stage3"]
    print(f"\n=== Stage 3: Ontology Mapping ===")
    print(f"Mapped frames: {len(stage3.get('mapped_frames', []))}")
    print(f"Ontology individuals: {len(stage3.get('ontology_individuals', []))}")
    print(f"Object property assertions: {len(stage3.get('object_property_assertions', []))}")
    print(f"Datatype property assertions: {len(stage3.get('datatype_property_assertions', []))}")
    
    # Show mapping audit for first frame
    if stage3.get("mapped_frames"):
        audit = stage3["mapped_frames"][0].get("mapping_audit", {})
        print(f"\nMapping audit (frame 0):")
        print(f"  Frame type mapped: {audit.get('frame_type_mapping')}")
        print(f"  Slot mappings: {audit.get('slot_mappings')}")
        print(f"  Unmapped slots: {audit.get('unmapped_slots')}")

# Enable debug logging for LLM calls
import logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger("graphrag_stage1.llm")
logger.setLevel(logging.DEBUG)

## 9. Useful Patterns & Tips

### Custom LLM Client
Implement the `LLMClient` protocol for any provider:
```python
from graphrag_stage1.llm import LLMClient

class MyCustomClient:
    model_id = "my-org/my-model-v1"
    
    def complete(self, prompt: str, schema: dict, *, stronger: bool = False) -> dict:
        # Call your API with JSON schema constraint
        response = my_api.call(prompt, schema=schema, model="larger" if stronger else "default")
        return response.parsed_json

client = MyCustomClient()
```

### Streaming / Checkpointing
```python
def checkpoint_callback(index: int, result: dict):
    # Save progress after each paragraph
    with open(f"checkpoint_para_{index}.json", "w") as f:
        json.dump(result, f)

results = run_paper(paragraphs, client=client, on_result=checkpoint_callback)
```

### Error Handling
```python
results = run_paper(paragraphs, client=client)
for r in results:
    if "error" in r:
        print(f"Failed: {r['paragraph_id']} - {r['error_type']}: {r['error']}")
    else:
        # Process success
        pass
```

### Provenance Tracking
Every proposition/frame carries `provenance` with:
- `document_id`, `source_uri`
- `page`, `char_start`, `char_end`
- `source_statement` (verbatim text span)

Use this for citation, verification, and human-in-the-loop review.

### Performance Tuning
| Parameter | Purpose | Typical Value |
|-----------|---------|---------------|
| `max_concurrency` | Global rate limit (provider quota) | 8-32 |
| `stage2_concurrency` | Per-paragraph frame fan-out | 4-8 |
| `BoundedClient` | Wraps client to enforce limits | Auto-applied |

### Output Formats
| Format | Use Case | Function |
|--------|----------|----------|
| JSON-LD | Graph DB import (Neptune, GraphDB) | `build_jsonld(stage3)` |
| Turtle (.ttl) | SPARQL endpoints, reasoning | `serialize_to_turtle(stage3)` |
| N-Quads | Bulk loading | `serialize_to_nquads(stage3)` |
| Python dicts | In-memory analysis | Direct access to `stage1`, `stage2`, `stage3` |

---

## Next Steps

1. **Add your domain ontology** to `ontologies/Domain/your_domain.owl`
2. **Extend alignment** in `ontologies/Alignment/stage2-alignment-v1.0.ttl` for custom frame types
3. **Run evaluation** on OSTI corpus: `python eval.py --corpus osti_corpus.json`
4. **Deploy batch pipeline** with `pipeline_runner.py` for production processing

---

## Resources

- **Design rationale**: [DESIGN.md](../DESIGN.md)
- **Integration guide**: [INTEGRATION.md](../INTEGRATION.md)
- **API reference**: `help(graphrag_stage1)` or `dir(graphrag_stage1)`
- **Example scripts**: `examples/quickstart.py`, `examples/model_clients.py`